# Homework 2 | Exploring Options | Tasks 1-4 due 09/22/25 | Task 5 due 09/24/25

Now that you have all of the options data stored locally on your computer and in pkl files, we can begin working with it. Your task will be to edit the monthly data and make some observations about its structure. See below.

## Tasks

### 1. For each file, remove the following columns: ['ImpliedVolatility', 'Delta','Gamma', 'Vega', 'Theta']

### 2. Add a column for the SPX index. MAKE SURE THE DATES MATCH! Hint: You might want to set the index of the options data to the date column... If yFinance doesn't let you download the SPX data skip this for now.

### 2. On paper or using markdown, use your knowledge of calculus to compute explicit formulas for the Greeks we discussed

### 3. Add columns of the Greeks using your formulae

### 4. Using the Newton Raphson method discussed on 9/17/25, calculate implied volatilies for each option and add this as a column to the data

### 5. Now plot the following and keep an eye out for specific relationships. We will talk about what we notice.

### 5.1 Using the ATM Strike and fixed date, plot the following:
* Call delta vs. time to maturity
* Call gamma vs. time to maturity
* Call theta vs. time to maturity
* Call vega vs. time to maturity
* Call implied volatility vs. time to maturity

Do the same for puts. What do you notice about the greeks for calls and puts?

### 5.2 From the span of 2015-2020:
* Calculate a 5,21,63,129 rolling realiezd volatilities for SPX
* Plot the implied volatilties for ATM options expiring at roughly the same 5,21,63,129 date marks. What I mean by this is, iterate through the data and calculate the implied volatility for ATM calls expriing in those times for every single day. You won't be using the same option for the 5 years if you get what I mean... Maybe it's clear already, and I'm being dramatic.
* Make some plots of IV - RV. What do you notice? When does the graph become positive?

### 5.3 Pick a date and maturity of your own choice and plot the following:
* Strike vs. delta
* Strike vs. theta
* Strike vs. IV
* Strike vs. option price

Feel free to do any additional analysis with the data at any time by the way. We are getting familiar with how options work here!

## Your work starts here

In [49]:
import pandas as pd
from pathlib import Path

pkl_path = Path("/Users/rayanarya/TAMID/HW02/pkl_files")

dfs = {}
for file in pkl_path.glob("*.pkl"):
    df=pd.read_pickle(file)
    dfs[file.stem] = df

len(dfs), list(dfs.keys())[:5]

(334,
 ['IVYOPPRCD_199811',
  'IVYOPPRCD_199805',
  'IVYOPPRCD_201602',
  'IVYOPPRCD_200508',
  'IVYOPPRCD_200905'])

In [50]:
cols_to_remove = ['ImpliedVolatility', 'Delta', 'Gamma', 'Theta', 'Vega']

for name, df in dfs.items():
    dfs[name] = df.drop(columns=[c for c in cols_to_remove if c in df.columns])

In [51]:
import yfinance as yf

spx = yf.download("^SPX", start="2010-01-01", end="2025-12-31")["Close"]
spx.name = "SPX"

for name, df in dfs.items():
    if "date" in df.columns:
        df = df.set_index("date")
    df = df.join(spx, how="left")
    dfs[name] = df.reset_index()
    

/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_3199/3119420263.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spx = yf.download("^SPX", start="2010-01-01", end="2025-12-31")["Close"]
[*********************100%***********************]  1 of 1 completed


In [60]:
import numpy as np
from scipy.stats import norm
import pandas as pd
import os
def delta_vec(S, K, T, r, sigma, option_type):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return np.where(option_type.str.upper() == 'C', norm.cdf(d1), norm.cdf(d1) - 1)

def gamma_vec(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

def theta_vec(S, K, T, r, sigma, option_type):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    first = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
    call_theta = first - r * K * np.exp(-r*T) * norm.cdf(d2)
    put_theta  = first + r * K * np.exp(-r*T) * norm.cdf(-d2)
    return np.where(option_type.str.upper() == 'C', call_theta, put_theta) / 365

def vega_vec(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return (S * norm.pdf(d1) * np.sqrt(T)) / 100


In [61]:
r= 0.04
sigma = 0.4

folder = r"/Users/rayanarya/TAMID/HW02/pkl_files"

def delta(S, K, T, r, sigma, option_type = 'C'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    if option_type == 'C':
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1
def gamma(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    pdf_d1 = norm.pdf(d1)
    gamma = pdf_d1 / (S * sigma * np.sqrt(T))
    return gamma
def theta(S, K, T, r, sigma, option_type = 'C'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    first = - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))

    if option_type == 'C':
        theta_val = first - r * K * np.exp(-r * T) * norm.cdf(d2)
    else:  # put
        theta_val = first + r * K * np.exp(-r * T) * norm.cdf(-d2)

    return theta_val / 365
def vega(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T)

    return vega / 100




def bs_price(S,K,T,r,sigma,option_type='C'):
    if T<=0 or S<=0 or K<=0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'C':
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    else: #Put
        return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
        
def bs_vega(S,K,T,r,sigma):
    if T<=0 or S<=0 or K<=0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return S*norm.pdf(d1)*np.sqrt(T)

def implied_vol_newton(S, K, T, r, market_price, option_type='C', sigma=0.4, tol=1e-6, max_iterations=100):
    if T<=0 or S<=0 or K<=0 or market_price<=0:
        return np.nan

    for _ in range(max_iterations):
     model_price = bs_price(S, K, T, r, sigma, option_type)
     diff = model_price - market_price
     if abs(diff) < tol:
        return sigma
        vega = bs_vega(S, K, T, r, sigma)
        if vega < 1e-8:
            return np.nan
        sigma = sigma - diff/vega
        sigma = np.clip(sigma, 1e-6, 5.0)
    return np.nan


for file in os.listdir(folder):
    if file.endswith(".pkl"):
        file_path = os.path.join(folder, file)

        df = pd.read_pickle(file_path)

        df["SPX"] = pd.to_numeric(df["SPX"], errors="coerce")
        df["Strike"] = pd.to_numeric(df["Strike"], errors="coerce")
        df["Strike"] = df["Strike"] / 1000  
        df["Expiration"] = pd.to_datetime(df["Expiration"], format="%Y%m%d", errors="coerce")
        df["T"] = (df["Expiration"] - df.index).dt.days / 365

        if "BestBid" in df.columns and "BestOffer" in df.columns:
            df["MidPrice"] = (df["BestBid"] + df["BestOffer"]) / 2

        df["Delta"] = delta_vec(df["SPX"], df["Strike"], df["T"], r, sigma, df["CallPut"])
        df["Gamma"] = gamma_vec(df["SPX"], df["Strike"], df["T"], sigma)
        df["Theta"] = theta_vec(df["SPX"], df["Strike"], df["T"], r, sigma, df["CallPut"])
        df["Vega"]  = vega_vec(df["SPX"], df["Strike"], df["T"], sigma)

        df["ImpliedVol"] = df.apply(
            lambda row: implied_vol_newton(
                row["SPX"], row["Strike"], max(row["T"], 1e-6), r, row["MidPrice"], row["CallPut"]
            ),
            axis=1
        )

        print(f"Loaded: {file_path}")
        print(df.head(10))

KeyError: 'SPX'

In [44]:
print(df.columns)


Index(['Date', 'index', 'SecurityID', 'symbol', 'symbolflag', 'Strike',
       'Expiration', 'CallPut', 'BestBid', 'BestOffer', 'LastTradeDate',
       'Volume', 'OpenInterest', 'SpecialSettlement', 'OptionID2',
       'AdjustmentFactor', 'AMSettlement', 'ContractSize', 'ExpiryIndicator',
       '^SPX', 'T', 'SPX', 'Delta', 'Gamma', 'Theta', 'Vega'],
      dtype='object')


In [ ]:
import matplotlib.pyplot as plt

def plot_greeks_vs_maturity(df, option_type='C'):
    # Select ATM options (closest strike to SPX each row)
    df['ATM'] = abs(df['Strike'] - df['SPX'])
    atm_df = df.loc[df.groupby('Date')['ATM'].idxmin()]  # pick closest strike per day

    # Filter by option type
    atm_df = atm_df[atm_df['CallPut'] == option_type]

    plt.figure(figsize=(12,8))
    plt.subplot(2,3,1); plt.scatter(atm_df['T'], atm_df['Delta'], alpha=0.5); plt.title(f"{option_type} Delta vs T")
    plt.subplot(2,3,2); plt.scatter(atm_df['T'], atm_df['Gamma'], alpha=0.5); plt.title(f"{option_type} Gamma vs T")
    plt.subplot(2,3,3); plt.scatter(atm_df['T'], atm_df['Theta'], alpha=0.5); plt.title(f"{option_type} Theta vs T")
    plt.subplot(2,3,4); plt.scatter(atm_df['T'], atm_df['Vega'], alpha=0.5); plt.title(f"{option_type} Vega vs T")
    plt.subplot(2,3,5); plt.scatter(atm_df['T'], atm_df['ImpliedVol'], alpha=0.5); plt.title(f"{option_type} IV vs T")
    plt.tight_layout()
    plt.show()

# Example: calls and puts
plot_greeks_vs_maturity(df, option_type='C')
plot_greeks_vs_maturity(df, option_type='P')


In [ ]:
# Compute log returns for SPX
spx_returns = np.log(spx / spx.shift(1))

# Rolling realized volatility (annualized, sqrt(252))
windows = [5,21,63,129]
rv = {w: spx_returns.rolling(w).std() * np.sqrt(252) for w in windows}

plt.figure(figsize=(12,6))
for w, series in rv.items():
    plt.plot(series.index, series, label=f"RV {w}d")
plt.legend(); plt.title("Rolling Realized Vol (SPX)"); plt.show()

# ATM Implied vols over time
atm_iv = atm_df.groupby('Date')['ImpliedVol'].mean()
plt.figure(figsize=(12,6))
plt.plot(atm_iv.index, atm_iv, label="ATM IV", color='black')
for w, series in rv.items():
    plt.plot(series.index, series, label=f"RV {w}d", alpha=0.7)
plt.legend(); plt.title("ATM IV vs Realized Vol"); plt.show()

# IV - RV plots
for w, series in rv.items():
    diff = atm_iv - series
    plt.figure(figsize=(12,4))
    plt.plot(diff.index, diff, label=f"IV - RV {w}d")
    plt.axhline(0, color='red', linestyle='--')
    plt.legend(); plt.title(f"IV - RV ({w}d)"); plt.show()


In [ ]:
def plot_strike_slices(df, target_date, target_expiry, option_type='C'):
    slice_df = df[(df['Date']==target_date) & (df['Expiration']==target_expiry) & (df['CallPut']==option_type)]
    slice_df = slice_df.sort_values('Strike')

    plt.figure(figsize=(12,8))
    plt.subplot(2,2,1); plt.plot(slice_df['Strike'], slice_df['Delta']); plt.title("Delta vs Strike")
    plt.subplot(2,2,2); plt.plot(slice_df['Strike'], slice_df['Theta']); plt.title("Theta vs Strike")
    plt.subplot(2,2,3); plt.plot(slice_df['Strike'], slice_df['ImpliedVol']); plt.title("IV vs Strike")
    plt.subplot(2,2,4); plt.plot(slice_df['Strike'], slice_df['MidPrice']); plt.title("Price vs Strike")
    plt.tight_layout(); plt.show()

# Example usage
plot_strike_slices(df, pd.Timestamp("2018-06-01"), pd.Timestamp("2018-06-15"), option_type='C')
